## Querying GeoSpatial data on US Census using DuckDB



In [1]:
import duckdb


In [ ]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [3]:
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"

roads_path = f"/vsizip//vsicurl/{roads_url}/tl_2024_35006_roads.shp"
states_path = f"/vsizip//vsicurl/{states_url}/tl_2024_us_state.shp"


In [4]:
! curl -s -I {roads_url}


HTTP/2 200 
date: Wed, 18 Nov 2026 14:09:08 GMT
content-type: application/zip
content-length: 2425481
content-security-policy: default-src 'self' 'unsafe-inline' 'unsafe-eval' census.gov *.census.gov http://www.census.gov assets.adobedtm.com www.googletagmanager.com https://dpm.demdex.net https://cm.everesttech.net https://uscensusbureau.demdex.net dap.digitalgov.gov www.google-analytics.com https://static.cloudflareinsights.com; frame-ancestors 'self';
x-frame-options: SAMEORIGIN
x-content-type-options: nosniff
cache-control: public, max-age=14400
referrer-policy: strict-origin-when-cross-origin
last-modified: Fri, 27 Jun 2025 20:04:20 GMT
accept-ranges: bytes
vary: Accept-Encoding
x-xss-protection: 1; mode=block
strict-transport-security: max-age=31536000
age: 411689
expires: Wed, 18 Nov 2026 18:09:08 GMT
cf-cache-status: HIT
set-cookie: __cf_bm=i45ewAu_wXFcS0gadnMZRO9G1Ryj5PSpQiR6_MRpFAs-1795010948.1679173-1.0.1.1-KKgBBSQf8cWrDTYB5SXdrKR9_p28Kk3h7nk4VEw6NK4fibZqrBzIvW5VzSpF_HWwtMYmG

In [5]:
! curl -s -I {states_url}


HTTP/2 200 
date: Wed, 18 Nov 2026 14:09:08 GMT
content-type: application/zip
content-length: 9954307
content-security-policy: default-src 'self' 'unsafe-inline' 'unsafe-eval' census.gov *.census.gov http://www.census.gov assets.adobedtm.com www.googletagmanager.com https://dpm.demdex.net https://cm.everesttech.net https://uscensusbureau.demdex.net dap.digitalgov.gov www.google-analytics.com https://static.cloudflareinsights.com; frame-ancestors 'self';
x-frame-options: SAMEORIGIN
x-content-type-options: nosniff
cache-control: public, max-age=14400
referrer-policy: strict-origin-when-cross-origin
last-modified: Fri, 27 Jun 2025 20:04:34 GMT
accept-ranges: bytes
vary: Accept-Encoding
x-xss-protection: 1; mode=block
strict-transport-security: max-age=31536000
age: 263912
expires: Wed, 18 Nov 2026 18:09:08 GMT
cf-cache-status: HIT
set-cookie: __cf_bm=gOKp_C4E4tnMkTfU6yhm9NHGkj0hJrr6jOzSqyT624E-1795010948.330643-1.0.1.1-Ih_mM8f15.dCcJI7D6Y__yZ6gQMZjnw7o_QNL9MG1t7SCCv8a.8sn.iZX.ghG36mbS6RdL

In [6]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM st_read('{roads_path}') roads
    LEFT JOIN st_read('{states_path}') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )
"""
print(query)



    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM st_read('/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp') roads
    LEFT JOIN st_read('/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip/tl_2024_us_state.shp') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )



In [7]:
gdf_combined = con.sql(query).df()
gdf_combined


IOException: IO Error: GDAL Error (4): `/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp' does not exist in the file system, and is not recognized as a supported dataset name.

In [8]:
query = f"""
  SELECT count(1)
  FROM st_read('{roads_path}')
"""

con.execute(query)


IOException: IO Error: GDAL Error (4): `/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp' does not exist in the file system, and is not recognized as a supported dataset name.